# COVID-19 Worldwide Testing Data

**Pregunta:** ¿Qué países reportaron más casos positivos en relación a la cantidad de tests realizados?


In [ ]:
# Solo la primera vez: instala las librerías. Si ya las tenés, saltate esta celda.
%pip install --user pandas matplotlib

In [ ]:
# Task 1 y 2: importar pandas y cargar el CSV
import pandas as pd

data = pd.read_csv('tested_worldwide.csv')
data.head()

In [ ]:
# Task 3: entender los datos
print("Filas y columnas:", data.shape)
print()
print("Tipos de cada columna:")
print(data.dtypes)
print()
print("Valores faltantes por columna:")
print(data.isnull().sum())

In [ ]:
# Task 4: limpieza
# 1) Nos quedamos solo con las columnas que necesitamos
data = data[['Country_Region', 'positive', 'total_tested']]

# 2) Renombramos para que sean mas claras
data = data.rename(columns={
    'Country_Region': 'Country',
    'positive': 'Positive Cases',
    'total_tested': 'Total Tested'
})

# 3) Eliminamos las filas con datos faltantes
data = data.dropna()

# 4) Convertimos los numeros al tipo correcto
data['Positive Cases'] = data['Positive Cases'].astype(int)
data['Total Tested'] = data['Total Tested'].astype(int)

data.head()

In [ ]:
# Confirmamos que ya no quedan faltantes
print(data.isnull().sum())

In [ ]:
# Task 5: total de casos positivos por pais (top 10)
positive_by_country = data.groupby('Country')['Positive Cases'].sum().reset_index()
positive_by_country = positive_by_country.rename(columns={'Positive Cases': 'Total Positive Cases'})
positive_by_country = positive_by_country.sort_values('Total Positive Cases', ascending=False)
positive_by_country.head(10)

In [ ]:
# Task 6: total de tests por pais (top 10)
tests_by_country = data.groupby('Country')['Total Tested'].sum().reset_index()
tests_by_country = tests_by_country.rename(columns={'Total Tested': 'Total Tests'})
tests_by_country = tests_by_country.sort_values('Total Tests', ascending=False)
tests_by_country.head(10)

In [ ]:
# Task 7: top 3 paises con mayor ratio de positivos vs tests
merged = positive_by_country.merge(tests_by_country, on='Country')

# Evitamos dividir por cero
merged = merged[merged['Total Tests'] > 0]

# ratio de positivos sobre tests
merged['Positive Test Rate'] = merged['Total Positive Cases'] / merged['Total Tests']

merged = merged.sort_values('Positive Test Rate', ascending=False)
merged[['Country', 'Positive Test Rate']].head(3)

In [ ]:
# Task 8: grafico - top 3 paises por ratio positivos/tests
import matplotlib.pyplot as plt

top3 = merged.head(3)
top3.plot(x='Country', y='Positive Test Rate', kind='bar', legend=False)
plt.title('Top 3 paises: ratio de positivos vs tests')
plt.ylabel('Positive Test Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Grafico - top 10 paises con mas casos positivos
positive_by_country.head(10).plot(x='Country', y='Total Positive Cases', kind='bar', legend=False)
plt.title('Top 10 paises con mas casos positivos')
plt.ylabel('Total Positive Cases')
plt.tight_layout()
plt.show()

In [ ]:
# Grafico - top 10 paises con mas tests realizados
tests_by_country.head(10).plot(x='Country', y='Total Tests', kind='bar', legend=False)
plt.title('Top 10 paises con mas tests realizados')
plt.ylabel('Total Tests')
plt.tight_layout()
plt.show()

## Task 9: Conclusiones

**Que encontramos:** los paises con mayor ratio positivos/tests son los que, por cada test hecho, hallaron mas casos (posible senal de que testeaban poco o solo a sintomaticos).

**Limitaciones:**
- Muchos valores faltantes; al eliminarlos se reduce la muestra.
- Sumar columnas acumulativas por fecha infla los totales (no es un conteo epidemiologico exacto).
- No todos los paises reportan igual ni con la misma frecuencia.

**Proximos pasos:** usar el ultimo valor por pais en vez de la suma, normalizar por poblacion, y analizar la evolucion en el tiempo.
